# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-awaischohan/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 4 — CTR / Engagement Opportunity Scoring.**

I picked this lane because it maps to a concrete, cheap action (review a page's title, meta description and intent match) and because the starter data shows the current way of finding these pages does not prioritise anything. The starter reason code `low_ctr_visible_page` (≥500 impressions, position 1–20, CTR < 0.5%) flags about 4 out of 5 visible pages (section 3). A list that flags almost everything is not a review queue. CTR also depends heavily on position, so a single fixed CTR cut-off compares pages that should not be compared. Adjusting for position and volume is a real problem worth 7 weeks, not a one-line filter.

Why not the others, for now: Lane 2 (Refresh) is the starter pipeline's own lane, and its label (`trend_direction == "down"`) is a current-window bucket that covers 54% of pages, so I would mostly be reusing the starter work. Lane 3 (Clustering) gives profiles, not a ranked decision. Lane 1 (Signal analysis) is useful but stops short of an action list. I can switch lanes until the end of Week 4 if the warehouse data says otherwise.

In [ ]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/m-awaischohan/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # walk up to the repo root from wherever the kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows x columns:", df.shape)
print("Clients:", df["client_id"].nunique())

Rows x columns: (30000, 44)
Clients: 32


## 2. The question: decision, action, cost of a wrong call

**Question:** Among pages that already get meaningful search visibility, which ones capture noticeably fewer clicks than pages at a similar position with similar volume, and so should be reviewed first for title, meta description or intent-match changes?

- **Unit of analysis (grain):** one content item (page), using its trailing-90-day metrics. One row = one page.
- **Output:** a ranked list of pages with a position-adjusted CTR gap score, reason codes (e.g. high impressions, strong position, CTR well below its tier, enough volume to trust it) and a confidence label.
- **Decision it improves:** which pages a content editor or SEO strategist reviews first, given limited time (for example, 20 pages per week).
- **Action:** the reviewer checks the page's search snippet and query intent, then rewrites the title/meta, adjusts the intro to match intent, or marks it "monitor — gap is explained by something else".
- **Cost of a wrong call:**
  - *False positive* (page flagged but fine): wasted reviewer time, and a rewrite can lower CTR on a page that was working.
  - *False negative* (real under-performer missed): clicks keep leaking quietly; nobody notices because nothing alerts.
  - Since reviewer time is the bottleneck, false positives at the top of the list matter most, so I will judge the ranking with **precision@K** (K = reviewer capacity), not overall accuracy.
- **Why data / ML helps, and why this is not just "train a model":** a plain CTR filter fails because CTR depends on position, impression volume, intent and content type. Low-volume pages have CTRs that swing on a single click. The core work is building a fair *expected* CTR for each page and ranking the gap; a model only earns its place if it beats a transparent position-tier baseline. The target must come from an **observed future outcome** (e.g. CTR in a later window), not from a rule I wrote, otherwise the model just relearns my rule.

**One-paragraph frame:** For a content reviewer deciding which pages to check first, I will build a ranked review queue from the starter data (later the warehouse release), scoring each page's position- and volume-adjusted CTR gap, measured by precision@K against a later-window outcome. A wrong call costs reviewer time or missed clicks. A fixed CTR rule is not enough because it ignores position and volume. I will claim only observed, decision-support results.

In [ ]:
# Grain check: one row should be one page
print("Unique content_id:", df["content_id"].nunique(), "of", len(df), "rows")
print("Duplicate content_id rows:", df["content_id"].duplicated().sum())

Unique content_id: 30000 of 30000 rows
Duplicate content_id rows: 0


## 3. Quick look at the data (2-3 real numbers)

Reminder from the data dictionary: `ctr` is already a percentage (`0.35` = 0.35%), and `avg_position = 0` means "no data", so those rows are excluded.

In [ ]:
# "Visible" = at least 500 impressions in 90 days and an average position between 1 and 20
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)]
flagged = visible[visible["ctr"] < 0.5]   # the starter's fixed low-CTR rule
print(f"Number 1 - visible pages: {len(visible):,}")
print(f"           flagged by the fixed 'CTR < 0.5%' rule: {len(flagged):,} ({len(flagged)/len(visible):.0%})")

# Number 2 - CTR changes with position (pooled clicks / impressions, pages with >= 500 impressions)
f = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)]
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
tier = (f.groupby("position_tier")[["clicks_90d", "impressions_90d"]].sum().reindex(order))
tier["pooled_ctr_pct"] = (100 * tier["clicks_90d"] / tier["impressions_90d"]).round(2)
tier["pages"] = f.groupby("position_tier").size().reindex(order)
print("\nNumber 2 - pooled CTR by position tier (>= 500 impressions):")
print(tier[["pages", "pooled_ctr_pct"]])

# Number 3 - pages far below their OWN tier's typical CTR
g = f[f["position_tier"].isin(["top_3", "page_1", "striking"])].copy()
g["tier_median_ctr"] = g.groupby("position_tier")["ctr"].transform("median")
gap = g[g["ctr"] < 0.5 * g["tier_median_ctr"]]
print(f"\nNumber 3 - pages below half their tier's median CTR: {len(gap):,} of {len(g):,} ({len(gap)/len(g):.0%})")
print(f"           share of those tiers' impressions they hold: {gap['impressions_90d'].sum()/g['impressions_90d'].sum():.0%}")

Number 1 - visible pages: 12,023
           flagged by the fixed 'CTR < 0.5%' rule: 9,759 (81%)

Number 2 - pooled CTR by position tier (>= 500 impressions):
               pages  pooled_ctr_pct
position_tier                       
top_3            458            0.49
page_1          7064            0.35
striking        4485            0.35
page_3_5        4330            0.16
deep             389            0.04

Number 3 - pages below half their tier's median CTR: 3,026 of 12,007 (25%)
           share of those tiers' impressions they hold: 18%


**What these numbers tell me:**

1. **12,023 pages are visible, and the fixed "CTR < 0.5%" rule flags 9,759 of them (81%).** A rule that flags four out of five pages cannot tell a reviewer where to start. This is the core gap my lane addresses.
2. **Pooled CTR falls from ~0.49% in the top 3 to ~0.35% on page 1, ~0.16% on pages 3–5 and ~0.04% deeper.** Position clearly shapes CTR, so pages must be compared within their position tier, not against one cut-off. Two things to investigate rather than trust yet: page 1 and "striking" (11–20) show almost the same pooled CTR, and the top-3 figure is far below the ≈2.78% the data dictionary reports at warehouse scale, so this slice may not be representative.
3. **A position-adjusted view narrows the list to 3,026 pages (25%) sitting below half their tier's median CTR, and they hold about 18% of those tiers' impressions.** That is a smaller, more focused pool with real exposure behind it — worth testing properly over the next weeks.

## 4. Careful words: what I can and can't claim

**What my work will be able to say:**
- *Observed:* which pages, in this data and time window, have CTR well below pages at a similar position and volume.
- *Directional:* which signals (position, volume, intent, content type, age) are associated with larger or smaller CTR gaps.
- *Decision-support:* a ranked list of pages that are reasonable candidates for review, with the reasons visible, and how well that ranking does on precision@K against a later-window outcome.

**What it will never be able to say:**
- That rewriting a title or meta description **will** raise CTR or **caused** a recovery. That needs an experiment; this data is observational.
- Anything about how Google's algorithm works or ranks pages.
- That low CTR always means a bad title. It can also come from SERP features, AI answers, brand vs non-brand query mix, or intent mismatch — the data cannot separate all of these.
- Results from low-volume pages. CTR on a page with a few dozen impressions moves a lot on one click, so I will set and state a minimum-volume floor.
- That results from the 30k starter slice hold for the full warehouse until I re-test there.

In [ ]:
# Why the volume floor matters: how much of the data is too thin to trust
print(f"Pages with < 100 impressions in 90 days: {(df['impressions_90d'] < 100).mean():.0%}")
print(f"Pages with zero clicks in 90 days:       {(df['clicks_90d'] == 0).mean():.0%}")
print(f"Pages with no position data (avg_position = 0): {(df['avg_position'] == 0).sum():,}")

Pages with < 100 impressions in 90 days: 27%
Pages with zero clicks in 90 days:       44%
Pages with no position data (avg_position = 0): 1,205
